# Bug Compass × Gemma 4: does structural context help a small local model find the bug?

This notebook reproduces the Gemma 4 experiment from the Bug Compass writeup.

1. **Score the committed runs** (CPU, under a minute).
2. **Optionally re-run Gemma 4** (`google/gemma-4-E4B-it-qat-q4_0-gguf`, Apache 2.0) with the same llama.cpp build on CPU. About 10 s per task per condition on 8 CPU threads in our run; use `LIMIT` for a quick check.

No gold edits are used in prompts or candidate pools. They are used only for scoring.

In [ ]:
!git clone -q https://github.com/Kulraj69/bug-compass
%cd bug-compass/gemma

## 1. Score the committed Gemma 4 runs

In [ ]:
!python evaluate.py

## 2. (Optional) Re-run Gemma 4 locally with llama.cpp
Set `LIMIT = 0` for all 300 tasks. Results are written to a separate folder so the committed runs are not overwritten.

In [ ]:
LIMIT = 20
import os, subprocess, time, urllib.request
os.makedirs('/kaggle/working/gemma', exist_ok=True)
!cd /kaggle/working/gemma && [ -f llama.tgz ] || curl -sL -o llama.tgz https://github.com/ggml-org/llama.cpp/releases/download/b11913/llama-b11913-bin-ubuntu-x64.tar.gz && tar xzf llama.tgz
!cd /kaggle/working/gemma && [ -f model.gguf ] || curl -sL -o model.gguf https://huggingface.co/google/gemma-4-E4B-it-qat-q4_0-gguf/resolve/main/gemma-4-E4B_q4_0-it.gguf
server = subprocess.Popen(['/kaggle/working/gemma/llama-b11913/llama-server','-m','/kaggle/working/gemma/model.gguf','-c','8192','--port','8088','--host','127.0.0.1','-np','1','--cache-ram','0'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(120):
    try:
        urllib.request.urlopen('http://127.0.0.1:8088/health'); break
    except Exception:
        time.sleep(2)
print('server up')

In [ ]:
import shutil
shutil.move('results', 'results_committed')
os.makedirs('results', exist_ok=True)
for cond in ('compass', 'plain'):
    !python rerank.py --cond {cond} --limit {LIMIT}
!python evaluate.py
server.terminate()

Expect small differences from the committed numbers if your llama.cpp build, CPU, or thread count differ. Temperature is 0 and thinking is disabled.